# Readme Data preprocessing pipeline

### 📌 Cell 1: Setup

In [1]:
#region Imports & Environment Setup
import os
import re
import time
import shutil
import json
import gzip
import hashlib
from pathlib import Path

# Set Working Directory & Path Configurations
SCRIPT_DIR = Path.cwd()

# Path Configurations
DOCS_DIR = (SCRIPT_DIR.parent / 'doc').resolve()
OUTPUT_JSON = (SCRIPT_DIR.parent / 'rag_chunks.json').resolve()
OUTPUT_GZ = (SCRIPT_DIR.parent / 'rag_chunks.json.gz').resolve()

# Pipeline Sizing Parameters
MIN_BODY_CHARS = 100          # Min body length to keep file (deletes stubs)
TARGET_CHUNK_SIZE = 250       # High-precision small child chunk size (~250 chars) for Vector Search
PARENT_MAX_CHARS = 2000       # Maximum Parent Context size for LLM Prompt generation
MIN_CHUNK_CHARS = 40          # Ignore tiny trailing fragments

print(f"[Config] Working Directory : {SCRIPT_DIR}")
print(f"[Config] Target Docs Dir    : {DOCS_DIR}")
print(f"[Config] Output JSON File   : {OUTPUT_JSON}")
print(f"[Config] Output GZ File     : {OUTPUT_GZ}")
#endregion

[Config] Working Directory : c:\Users\boyce\OneDrive\Desktop\Enterprise-RAG-Engine\preprocessing-pipeline\script
[Config] Target Docs Dir    : C:\Users\boyce\OneDrive\Desktop\Enterprise-RAG-Engine\preprocessing-pipeline\doc
[Config] Output JSON File   : C:\Users\boyce\OneDrive\Desktop\Enterprise-RAG-Engine\preprocessing-pipeline\rag_chunks.json
[Config] Output GZ File     : C:\Users\boyce\OneDrive\Desktop\Enterprise-RAG-Engine\preprocessing-pipeline\rag_chunks.json.gz


### 📌 Cell 2: Phase 1 — Clean
Using regrex to clean up the scraped md file in DOCS_DIR

In [ ]:
#region Phase 1 - Integrated Markdown Cleaning Engine
import re
from pathlib import Path

# Environment & Path Configuration (Independent Execution Support)
SCRIPT_DIR = Path.cwd()
DOCS_DIR = (SCRIPT_DIR.parent / 'doc').resolve()
MIN_BODY_CHARS = 100

_FM_NAV_JUNK = re.compile(r"\s*[-–]?\s*Kanzi framework[\d\s.]*documentation.*$", re.IGNORECASE)
_FM_NAV_EXTRAS = re.compile(r"(ContentsMenuExpand|Light mode|Dark mode|Auto light/dark[^\n]*)")
_IMAGE_MD = re.compile(r"!\[[^\]]*\]\([^)]*_images/[^)]+\)")
_NOISE_SECTION_HEADINGS = re.compile(r"^#{1,6}\s+(See also|Prerequisites|Related topics|In this section|On this page|Contents|Navigation|Next steps?)\s*$", re.IGNORECASE | re.MULTILINE)
_REL_LINK = re.compile(r"\[([^\]]+)\]\((?!https?://)([^)]+\.html[^)]*)\)")
_URL_DUP = re.compile(r"\[(https?://[^\]]+)\]\(\1\)")
_CALLOUT_HEADING = re.compile(r"^(#{1,6})\s+(Tip|Note|Warning|Important|Caution)\s*$", re.IGNORECASE)
_BROKEN_LIST_ITEM = re.compile(r"^-\s*$\n\n(?=\S)", re.MULTILINE)

def flatten_markdown_tables(text: str) -> str:
    """Converts Markdown tables into structured Key: Value natural language lines for vector search optimization."""
    lines = text.split("\n")
    flattened_lines = []
    in_table = False
    headers = []
    for line in lines:
        stripped = line.strip()
        if "|" in line and stripped.startswith("|") and stripped.endswith("|"):
            parts = [p.strip() for p in line.split("|")[1:-1]]
            if not parts or all(re.match(r"^:?-+:?$", p) for p in parts):
                continue
            if not in_table:
                in_table = True
                headers = parts
            else:
                row_items = [f"{h}: {v}" for h, v in zip(headers, parts) if v and h]
                if row_items:
                    flattened_lines.append("- " + ", ".join(row_items))
        else:
            in_table = False
            headers = []
            flattened_lines.append(line)
    return "\n".join(flattened_lines)

def clean_markdown_text(text: str) -> str:
    """Applies text cleaning rules to raw Markdown text."""
    text = text.replace("Â¶", "").replace("¶", "")
    lines = text.split("\n")
    fm_clean = []
    in_fm, fm_count = False, 0
    for line in lines:
        if line.rstrip() == "---":
            fm_count += 1
            in_fm = (fm_count == 1)
            fm_clean.append(line)
            continue
        if in_fm and line.startswith("title:"):
            title = line[len("title:"):].strip()
            title = _FM_NAV_JUNK.sub("", title).strip()
            title = _FM_NAV_EXTRAS.sub("", title).strip().rstrip(" -").strip()
            fm_clean.append(f"title: {title}")
        else:
            fm_clean.append(line)
    text = "\n".join(fm_clean)

    text = _IMAGE_MD.sub("", text)
    lines = text.split("\n")
    res_lines = []
    skip_level = None
    for line in lines:
        hm = re.match(r"^(#{1,6})\s+(.*)", line)
        if hm:
            level = len(hm.group(1))
            if skip_level is not None:
                if level <= skip_level:
                    skip_level = None
                else:
                    continue
            if _NOISE_SECTION_HEADINGS.match(line):
                skip_level = level
                continue
        elif skip_level is not None:
            continue
        res_lines.append(line)
    text = "\n".join(res_lines)

    text = _REL_LINK.sub(r"\1", text)
    text = _URL_DUP.sub(r"\1", text)
    lines = text.split("\n")
    co_lines = []
    for line in lines:
        m = _CALLOUT_HEADING.match(line)
        if m:
            kind = m.group(2).capitalize()
            co_lines.append(f"> **{kind}:**")
        else:
            co_lines.append(line)
    text = "\n".join(co_lines)

    text = _BROKEN_LIST_ITEM.sub("- ", text)
    text = re.sub(r"^\s*>\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*>\s?", "", text, flags=re.MULTILINE)
    text = flatten_markdown_tables(text)

    lines = [l.rstrip() for l in text.split("\n")]
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n"

print("=== Starting Phase 1: Text Cleaning & Noise Elimination ===")
all_md_files = sorted(f for f in DOCS_DIR.rglob("*.md") if f.suffix == ".md")
cleaned_count, purged_stubs = 0, 0
for md_file in all_md_files:
    raw_text = md_file.read_text(encoding="utf-8", errors="replace")
    cleaned_text = clean_markdown_text(raw_text)
    if len(cleaned_text) < MIN_BODY_CHARS:
        md_file.unlink()
        purged_stubs += 1
    else:
        md_file.write_text(cleaned_text, encoding="utf-8")
        cleaned_count += 1

print(f"Phase 1 Complete: Cleaned {cleaned_count} Markdown files, Purged {purged_stubs} stubs.")
#endregion

=== Starting Phase 1: Text Cleaning & Noise Elimination ===
Phase 1 Complete: Cleaned 0 Markdown files, Purged 0 stubs.


### 📌 Cell 3: Phase 2 — VDB Semantic Chunking Engine & Dataset Export
Parses document structure, performs H1/H2/H3 sectioning, separates code/prose, prepends Contextual Headers, links parent content, and exports compressed JSON dataset.

In [ ]:
#region Phase 2 - Semantic Chunking & Dataset Export
import re
import json
import gzip
import hashlib
from pathlib import Path

# Environment & Path Configuration (Independent Execution Support)
SCRIPT_DIR = Path.cwd()
DOCS_DIR = (SCRIPT_DIR.parent / 'doc').resolve()
OUTPUT_JSON = (SCRIPT_DIR.parent / 'rag_chunks.json').resolve()
OUTPUT_GZ = (SCRIPT_DIR.parent / 'rag_chunks.json.gz').resolve()
TARGET_CHUNK_SIZE = 250
PARENT_MAX_CHARS = 2000
MIN_CHUNK_CHARS = 40

_FRONTMATTER_RE = re.compile(r"^---\s*\n(.*?)\n---\s*\n", re.DOTALL)
heading_pattern = re.compile(r"^(#{1,3})\s+(.+)$", re.MULTILINE)

def _flush_prose_accumulator(acc_list, section_path, page_title, source_url, rel_path, parent_id, parent_content, seen_hashes, chunks):
    """Flushes accumulated prose paragraphs into a contextualized chunk object and appends to chunk list if unique."""
    if not acc_list:
        return
    body_text = "\n\n".join(acc_list)
    ctx_prefix = f"[Document Context: {section_path}]" if section_path else f"[Document Context: {page_title}]"
    full_content = f"{ctx_prefix}\n\n{body_text}"
    c_hash = hashlib.sha256(full_content.encode("utf-8")).hexdigest()

    if c_hash not in seen_hashes:
        seen_hashes.add(c_hash)
        chunks.append({
            "id": f"{rel_path}#chunk-{len(seen_hashes)}",
            "content": full_content,
            "chunk_type": "prose",
            "metadata": {
                "source_url": source_url,
                "local_path": rel_path,
                "page_title": page_title,
                "section_path": section_path,
                "chunk_type": "prose",
                "parent_id": parent_id,
                "parent_content": parent_content[:PARENT_MAX_CHARS],
                "context_enriched": True,
                "char_count": len(full_content),
                "hash": c_hash,
                "questions": []
            }
        })

def _subchunk_prose(prose_text, section_path, page_title, source_url, rel_path, parent_id, parent_content, seen_hashes):
    """Splits a prose text block into smaller child chunks constrained by TARGET_CHUNK_SIZE."""
    chunks = []
    paragraphs = [p.strip() for p in prose_text.split("\n\n") if p.strip()]
    current_acc = []
    current_len = 0

    for para in paragraphs:
        if current_len + len(para) > TARGET_CHUNK_SIZE and current_acc:
            _flush_prose_accumulator(current_acc, section_path, page_title, source_url, rel_path, parent_id, parent_content, seen_hashes, chunks)
            current_acc = [para]
            current_len = len(para)
        else:
            current_acc.append(para)
            current_len += len(para)

    _flush_prose_accumulator(current_acc, section_path, page_title, source_url, rel_path, parent_id, parent_content, seen_hashes, chunks)
    return chunks

def process_file(filepath, docs_dir, seen_hashes):
    """Parses a single Markdown file into semantic sections, separates code/prose, and generates enriched vector chunks."""
    rel_path = filepath.relative_to(docs_dir).as_posix()
    html_rel = rel_path.replace(".md", ".html")
    source_url = f"https://docs.kanzi.com/4.1.0/en/{html_rel}"

    raw_text = filepath.read_text(encoding="utf-8", errors="replace")
    page_title = filepath.stem.replace("-", " ").title()

    fm_match = _FRONTMATTER_RE.match(raw_text)
    body = raw_text
    if fm_match:
        body = raw_text[fm_match.end():]
        fm_text = fm_match.group(1)
        for line in fm_text.splitlines():
            if line.startswith("title:"):
                page_title = line[len("title:"):].strip()
                break

    matches = list(heading_pattern.finditer(body))
    sections = []
    if not matches:
        sections.append({"h_level": 1, "heading": page_title, "section_path": page_title, "content": body.strip()})
    else:
        current_h1, current_h2, current_h3 = page_title, "", ""
        preamble = body[:matches[0].start()].strip()
        if preamble:
            sections.append({"h_level": 1, "heading": page_title, "section_path": page_title, "content": preamble})

        for idx, match in enumerate(matches):
            level = len(match.group(1))
            heading_text = match.group(2).strip()
            start_pos = match.end()
            end_pos = matches[idx + 1].start() if idx + 1 < len(matches) else len(body)
            section_content = body[start_pos:end_pos].strip()
            if level == 1:
                current_h1, current_h2, current_h3 = heading_text, "", ""
            elif level == 2:
                current_h2, current_h3 = heading_text, ""
            elif level == 3:
                current_h3 = heading_text
            path_parts = [p for p in [current_h1, current_h2, current_h3] if p]
            section_path = " > ".join(path_parts)
            if section_content:
                sections.append({"h_level": level, "heading": heading_text, "section_path": section_path, "content": section_content})

    chunks = []
    for sec_idx, sec in enumerate(sections, start=1):
        section_path = sec.get("section_path", page_title)
        sec_content = sec["content"]
        parent_id = f"{rel_path}#sec-{sec_idx}"
        ctx_prefix = f"[Document Context: {section_path}]" if section_path else f"[Document Context: {page_title}]"
        parent_content = f"{ctx_prefix}\n\n{sec_content}"

        code_block_pattern = re.compile(r"```(\w*)\n(.*?)```", re.DOTALL)
        last_end = 0
        for cb_match in code_block_pattern.finditer(sec_content):
            cb_start, cb_end = cb_match.span()
            prose_part = sec_content[last_end:cb_start].strip()
            code_lang = cb_match.group(1) or "text"
            code_content = cb_match.group(2).strip()
            if len(prose_part) >= MIN_CHUNK_CHARS:
                chunks.extend(_subchunk_prose(prose_part, section_path, page_title, source_url, rel_path, parent_id, parent_content, seen_hashes))
            if len(code_content) >= 15:
                code_text = f"{ctx_prefix}\n```{code_lang}\n{code_content}\n```"
                c_hash = hashlib.sha256(code_text.encode("utf-8")).hexdigest()
                if c_hash not in seen_hashes:
                    seen_hashes.add(c_hash)
                    chunks.append({
                        "id": f"{rel_path}#code-{len(chunks)+1}",
                        "content": code_text,
                        "chunk_type": "code",
                        "code_lang": code_lang,
                        "metadata": {
                            "source_url": source_url,
                            "local_path": rel_path,
                            "page_title": page_title,
                            "section_path": section_path,
                            "chunk_type": "code",
                            "code_lang": code_lang,
                            "parent_id": parent_id,
                            "parent_content": parent_content[:PARENT_MAX_CHARS],
                            "context_enriched": True,
                            "char_count": len(code_text),
                            "hash": c_hash,
                            "questions": []
                        }
                    })
            last_end = cb_end
        remaining_prose = sec_content[last_end:].strip()
        if len(remaining_prose) >= MIN_CHUNK_CHARS:
            chunks.extend(_subchunk_prose(remaining_prose, section_path, page_title, source_url, rel_path, parent_id, parent_content, seen_hashes))
    return chunks

print("=== Starting Phase 2: Semantic Chunking & Dataset Building ===")
md_files = sorted(f for f in DOCS_DIR.rglob("*.md") if f.suffix == ".md")
print(f"  Processing {len(md_files)} clean Markdown source files...")

all_chunks, seen_hashes = [], set()
code_count, prose_count = 0, 0
for filepath in md_files:
    file_chunks = process_file(filepath, DOCS_DIR, seen_hashes)
    for c in file_chunks:
        if c["chunk_type"] == "code":
            code_count += 1
        else:
            prose_count += 1
    all_chunks.extend(file_chunks)

dataset_payload = {
    "dataset_name": "kanzi-framework-4.1.0-rag-chunks",
    "doc_version": "4.1.0",
    "total_source_files": len(md_files),
    "total_chunks": len(all_chunks),
    "prose_chunks_count": prose_count,
    "code_chunks_count": code_count,
    "chunks": all_chunks
}

OUTPUT_JSON.write_text(json.dumps(dataset_payload, indent=2, ensure_ascii=False), encoding="utf-8")
with gzip.open(OUTPUT_GZ, "wb") as f_gz:
    f_gz.write(json.dumps(dataset_payload, indent=2, ensure_ascii=False).encode("utf-8"))

size_mb = OUTPUT_JSON.stat().st_size / (1024 * 1024)
size_gz_mb = OUTPUT_GZ.stat().st_size / (1024 * 1024)
print(f"\nPhase 2 Complete:")
print(f"  - Total Chunks Exported : {len(all_chunks)}")
print(f"  - Prose Chunks          : {prose_count}")
print(f"  - Code Chunks           : {code_count}")
print(f"  - Output JSON Payload   : {OUTPUT_JSON.name} ({size_mb:.2f} MB)")
print(f"  - Output GZ Payload     : {OUTPUT_GZ.name} ({size_gz_mb:.2f} MB)")
#endregion

=== Starting Phase 2: Semantic Chunking & Dataset Building ===
  Processing 0 clean Markdown source files...

Phase 2 Complete:
  - Total Chunks Exported : 0
  - Prose Chunks          : 0
  - Code Chunks           : 0
  - Output JSON Payload   : kanzi_rag_chunks.json (0.00 MB)
  - Output GZ Payload     : kanzi_rag_chunks.json.gz (0.00 MB)


### 📌 Cell 4: Phase 3 — [L] Zero-Cost Local Hypothetical Question Engine
Generates 2–3 targeted hypothetical natural language questions for every chunk without external API calls or fees ($0.00 cost).
Uses document hierarchy, section semantic analysis, code block metadata, and action verb matching to construct realistic user search patterns.

In [7]:
#region Phase 3 - [L] Zero-Cost Local Hypothetical Question Engine
import re
import json
import gzip
from pathlib import Path

# Environment & Path Configuration (Independent Execution Support)
SCRIPT_DIR = Path.cwd()
OUTPUT_JSON = (SCRIPT_DIR.parent / 'rag_chunks.json').resolve()
OUTPUT_GZ = (SCRIPT_DIR.parent / 'rag_chunks.json.gz').resolve()

ACTION_VERBS = [
    "configure", "creating", "create", "setting", "set", "using", "use",
    "enabling", "enable", "disabling", "disable", "connecting", "connect",
    "adding", "add", "removing", "remove", "optimizing", "optimize",
    "rendering", "render", "animating", "animate", "binding", "bind"
]

def extract_hypothetical_questions(chunk: dict) -> list[str]:
    """Generates 2-3 hypothetical natural language search questions for a chunk based on section context and action verbs."""
    meta = chunk.get("metadata", {})
    title = meta.get("page_title", "").strip()
    section = meta.get("section_path", "").strip()
    chunk_type = meta.get("chunk_type", "prose")
    code_lang = meta.get("code_lang", "none")
    content = chunk.get("content", "")

    questions = []
    sec_parts = [p.strip() for p in section.split(">") if p.strip()]
    leaf_section = sec_parts[-1] if sec_parts else title
    parent_section = sec_parts[-2] if len(sec_parts) >= 2 else ""

    if chunk_type == "code":
        lang_str = f"in {code_lang}" if code_lang and code_lang != "none" else ""
        if parent_section and leaf_section:
            questions.append(f"How to implement {leaf_section} for {parent_section} {lang_str}?".strip())
            questions.append(f"Code example for {leaf_section} {lang_str}".strip())
        elif leaf_section:
            questions.append(f"How to write {leaf_section} code {lang_str}?".strip())
            questions.append(f"Example code for {leaf_section} {lang_str}".strip())
        else:
            questions.append(f"Code snippet example for {title} {lang_str}".strip())
    else:
        matched_action = None
        for verb in ACTION_VERBS:
            if re.search(r'\b' + verb + r'\b', content, re.IGNORECASE):
                matched_action = verb
                break

        if parent_section and leaf_section:
            questions.append(f"How to use {leaf_section} in {parent_section}?")
            questions.append(f"What is the function of {leaf_section} in {parent_section}?")
            if matched_action:
                questions.append(f"How to {matched_action} {leaf_section} in Kanzi?")
            else:
                questions.append(f"How does {leaf_section} work in Kanzi?")
        elif leaf_section:
            questions.append(f"What is {leaf_section} in Kanzi?")
            questions.append(f"How to configure {leaf_section}?")
            questions.append(f"Overview and usage of {leaf_section}")
        else:
            questions.append(f"What is {title}?")
            questions.append(f"How to work with {title} in Kanzi?")

    unique_q = []
    for q in questions:
        q_clean = q.strip()
        if q_clean and q_clean not in unique_q:
            unique_q.append(q_clean)
    return unique_q[:3]

print("=== Starting Phase 3: Zero-Cost Local Question Generation ===")
if "dataset_payload" not in dir() or dataset_payload is None:
    if OUTPUT_GZ.exists():
        with gzip.open(OUTPUT_GZ, "rt", encoding="utf-8") as f:
            dataset_payload = json.load(f)
    elif OUTPUT_JSON.exists():
        with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
            dataset_payload = json.load(f)
    else:
        dataset_payload = {"chunks": []}

chunks = dataset_payload.get("chunks", [])
for chunk in chunks:
    chunk.setdefault("metadata", {})["questions"] = extract_hypothetical_questions(chunk)

dataset_payload["chunks"] = chunks
payload = json.dumps(dataset_payload, indent=2, ensure_ascii=False)
OUTPUT_JSON.write_text(payload, encoding="utf-8")
with gzip.open(OUTPUT_GZ, "wb") as f_gz:
    f_gz.write(payload.encode("utf-8"))

size_mb = OUTPUT_JSON.stat().st_size / (1024 * 1024)
size_gz_mb = OUTPUT_GZ.stat().st_size / (1024 * 1024)
print(f"Phase 3 Complete (Zero API Cost):")
print(f"  - Questions populated : {len(chunks):,} / {len(chunks):,} (100% coverage)")
print(f"  - Output JSON Payload : {OUTPUT_JSON.name} ({size_mb:.2f} MB)")
print(f"  - Output GZ Payload     : {OUTPUT_GZ.name} ({size_gz_mb:.2f} MB)")
#endregion

=== Starting Phase 3: Zero-Cost Local Question Generation ===
Phase 3 Complete (Zero API Cost):
  - Questions populated : 0 / 0 (100% coverage)
  - Output JSON Payload : kanzi_rag_chunks.json (0.00 MB)
  - Output GZ Payload   : kanzi_rag_chunks.json.gz (0.00 MB)


### 📌 Cell 5: Phase 4 — Verification & Inspection
Loads the exported dataset to inspect Child Search Content, Parent LLM Context, and generated Hypothetical Questions.

In [5]:
#region Phase 4 Verification & Inspection
import json
import gzip
from pathlib import Path

# Environment & Path Configuration (Independent Execution Support)
SCRIPT_DIR = Path.cwd()
OUTPUT_JSON = (SCRIPT_DIR.parent / 'rag_chunks.json').resolve()
OUTPUT_GZ = (SCRIPT_DIR.parent / 'rag_chunks.json.gz').resolve()

if OUTPUT_GZ.exists():
    with gzip.open(OUTPUT_GZ, "rt", encoding="utf-8") as f:
        verify_data = json.load(f)
elif OUTPUT_JSON.exists():
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        verify_data = json.load(f)
else:
    verify_data = {"dataset_name": "N/A", "total_chunks": 0, "chunks": []}

print(f"Loaded Dataset Name : {verify_data.get('dataset_name', 'N/A')}")
print(f"Total Vector Chunks : {verify_data.get('total_chunks', 0)}")

if verify_data.get("chunks"):
    sample_chunk = verify_data["chunks"][0]
    print(f"\nSample Chunk Inspection (Chunk #1):")
    print(f"  [Chunk ID]        : {sample_chunk['id']}")
    print(f"  [Search Content]  : {sample_chunk['content'][:140]}...")
    print(f"  [Parent ID]       : {sample_chunk['metadata']['parent_id']}")
    print(f"  [Parent Context]  : {sample_chunk['metadata']['parent_content'][:180]}...")
    print(f"  [Questions (HQ)]  : {sample_chunk['metadata'].get('questions')}")

    hq_populated = sum(1 for c in verify_data["chunks"] if c.get("metadata", {}).get("questions"))
    tot = verify_data.get('total_chunks', len(verify_data["chunks"]))
    cov = (hq_populated / tot * 100) if tot > 0 else 0
    print(f"\nHypothetical Questions Coverage:")
    print(f"  - Chunks with questions : {hq_populated:,} / {tot:,}")
    print(f"  - Coverage              : {cov:.1f}%")
else:
    print("\nNo chunks available to inspect.")
#endregion

Loaded Dataset Name : Kanzi Documentation VDB Vector Chunks
Total Vector Chunks : 31626

Sample Chunk Inspection (Chunk #1):
  [Chunk ID]        : best-practices/animations/animations-best-practices.md#chunk-1
  [Search Content]  : [Document Context: Animations best practices]

To create more efficient animations:...
  [Parent ID]       : best-practices/animations/animations-best-practices.md#sec-1
  [Parent Context]  : [Document Context: Animations best practices]

To create more efficient animations:

- Remove the keyframes that do not affect the precision of an animation.
- Remove the Animation...
  [Questions (HQ)]  : ['What is Animations best practices in Kanzi?', 'How to configure Animations best practices?', 'Overview and usage of Animations best practices']

Hypothetical Questions Coverage:
  - Chunks with questions : 31,626 / 31,626
  - Coverage              : 100.0%
